## tl;dr

Walk-forward trên dữ liệu local cho thấy heuristic hiện tại chưa tạo lift ổn định so với baseline toán học: Power 6/55 đạt khoảng 0,653 số chính/kỳ so với 0,655 của vé ngẫu nhiên; XSMN cũng nằm trong khoảng nhiễu. Vì vậy thay đổi hợp lý là cải thiện đo lường, highlight kết quả và chỉ nâng model version khi backtest out-of-sample vượt baseline với khoảng tin cậy không cắt 0.

## Context & Methods

Notebook này là companion audit cho `analysis/prediction_backtest.py`. Mỗi dự đoán chỉ dùng các kỳ trước ngày cần dự đoán; 100 kỳ đầu mỗi đài/sản phẩm là warm-up. Baseline là kỳ vọng tổ hợp của một phép quay công bằng, không phải một chuỗi random duy nhất.

### Key Assumptions

- Các kỳ quay độc lập và mọi số hợp lệ có xác suất bằng nhau nếu hệ thống quay công bằng.
- Backtest mô tả hiệu năng lịch sử, không chứng minh khả năng dự báo tương lai.
- Dữ liệu gần nhất có thể khác ngày giữa SQLite local và prediction history; trường `as_of` phải được đọc trước khi diễn giải.

## Data

In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from analysis.prediction_backtest import run_backtest

results = run_backtest(ROOT)
results['as_of'], results['data_quality']

({'xsmn': '2026-07-12', 'power655': '2026-07-11'},
 {'xsmn': {'draws': 23155,
   'prize_rows': 416790,
   'first_date': '2005-12-29',
   'last_date': '2026-07-12',
   'stations': 21,
   'incomplete_draws': 0},
  'power655': {'draws': 1370,
   'first_date': '2017-08-01',
   'last_date': '2026-07-11',
   'missing_draw_ids': 0}})

## Results

In [2]:
power_models = pd.DataFrame(results['power655']['models'])
xsmn_special_models = pd.DataFrame(results['xsmn_special']['models'])
xsmn_full_models = pd.DataFrame(results['xsmn_full_prize']['models'])
display(power_models)
display(xsmn_special_models)
display(xsmn_full_models)

,model,evaluated_draws,mean_main_matches,mean_ci95,draws_with_at_least_two,draws_with_at_least_three,lift_vs_theoretical_random
0,current_v1,1270,0.652756,"[0.614173, 0.693701]",0.121260,0.011024,-0.002734
1,top6_blend,1270,0.641732,"[0.603937, 0.680315]",0.106299,0.011811,-0.019576
2,top6_without_gap,1270,0.648819,"[0.609449, 0.688976]",0.127559,0.011811,-0.008749
3,top6_long_term,1270,0.634646,"[0.59685, 0.673228]",0.112598,0.011024,-0.030402


,model,evaluated_draws,mean_matching_suffix_digits,mean_ci95,suffix_at_least_one_rate,suffix_at_least_two_rate,suffix_at_least_three_rate,exact_rate
0,current_v1,17314,0.106619,"[0.101536, 0.111702]",0.097031,0.008317,0.000982,0.000058
1,positional_map,17314,0.108814,"[0.103616, 0.113954]",0.098648,0.009068,0.000866,0.000000
2,suffix2_map,17314,0.110662,"[0.105348, 0.115918]",0.098475,0.010454,0.001559,0.000000
3,recent_map,17314,0.110777,"[0.105695, 0.116207]",0.099168,0.010338,0.001213,0.000000


,model,evaluated_draws,mean_exact_prize_hits,mean_ci95,draws_with_any_exact_hit,total_exact_hits,lift_vs_theoretical_random
0,current_v1,21055,0.013488,"[0.011921, 0.015056]",0.013393,284,0.074694
1,positional_map,21055,0.013251,"[0.011731, 0.014866]",0.013251,279,0.055773


In [3]:
pd.DataFrame(results['xsmn_full_prize']['by_prize'])

,prize_code,eligible_draws,random_expected_total_hits,current_v1_total_hits,positional_map_total_hits
0,g8,21055,210.550000,214,208
1,g7,21055,21.055000,30,24
2,g6,21055,18.949500,32,31
3,g5,21055,2.105500,1,4
4,g4,21055,10.316950,5,12
5,g3,21055,0.842200,1,0
6,g2,21055,0.210550,0,0
7,g1,21055,0.210550,0,0
8,db,19414,0.019414,1,0


## Takeaways

- Không có biến thể nào đủ bằng chứng để thay model hiện tại; chọn top score hoặc bỏ gap không tạo lift ổn định.
- `model_score` cần tiếp tục được gọi là điểm xếp hạng, không phải xác suất.
- Mọi model mới phải qua walk-forward, báo cả sample size, baseline tổ hợp, khoảng tin cậy và hiệu năng theo hạng giải.
- Cách duy nhất tăng xác suất toán học trực tiếp là tăng số vé/tổ hợp không trùng nhau; đó là tăng coverage và chi phí, không phải tăng kỹ năng dự báo.